# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their field ids
record_sets = [rs for rs in dataset.metadata.record_sets]
print(f"Record sets found: {[rs['@id'] for rs in record_sets]}")
print()
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    print("  Fields:")
    for f in rs['fields']:
        print(f"    - {f['@id']}: {f.get('name','N/A')} ({f.get('dataType','N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this notebook, we pick the FIRST record set for demonstration.
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
# We'll use the first record set (edit if you want to explore another one)
record_set_id = record_sets_ids[0]

# Load the records for each record set
dataframes = {}

for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Columns in {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, choose a numeric field present in the selected record set.
df = dataframes[record_set_id]
# Auto-detect numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_cols) == 0:
    print("No numeric columns found in the dataset!")
else:
    numeric_field = numeric_cols[0]
    print(f"Analyzing numeric field: {numeric_field}")

    # Set an arbitrary threshold (e.g., mean)
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the selected field
    field_norm_col = f"{numeric_field}_normalized"
    filtered_df[field_norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, field_norm_col]].head())

    # Attempt to group by a string/categorical field
    possible_group_cols = df.select_dtypes(include="object").columns.tolist()
    group_field = None
    for col in possible_group_cols:
        if col != numeric_field:
            group_field = col
            break
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable categorical field found to group by.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Visualize distribution of the numeric field
if len(numeric_cols) > 0:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30, color='steelblue', alpha=0.7)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group if a group field is available
    if group_field is not None:
        plt.figure(figsize=(10,5))
        filtered_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=45)
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric column to visualize!")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we successfully loaded and explored the FAIR<sup>2</sup> Croissant dataset “Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells” using the `mlcroissant` library.

- We inspected the structure by listing available record sets and their fields using `@id`.
- We loaded a record set and extracted its numeric and categorical fields.
- We demonstrated basic filtering, normalization, and group analysis.
- We visualized distributions and grouped statistics for deeper understanding.

You can further tailor the analysis by inspecting additional record sets or fields via their `@id`, and by extending the EDA/visualization sections. 